# Imports & Setup

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch import nn
import matplotlib.pyplot as plt
import os
import numpy as np
import imageio
from collections import OrderedDict
from typing import Dict, Callable

# Load Model and Tokenizer

In [8]:
model_name = "Qwen/Qwen3-4B"
# model_name = "openai-community/gpt2"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-4B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=F

In [10]:
# from transformers import AutoModelForCausalLM

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     dtype='auto',
#     device_map=device
# )

# print(model.device)
# print(model)

In [20]:
# Apply chat template to prompt
prompt = "The capital of France is"
messages = [
    {"role": "user", "content": prompt}
]
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)

# Tokenize inputs and move to device
inputs = tokenizer([tokenized_chat], return_tensors="pt").to(model.device)
inputs

{'input_ids': tensor([[151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
            198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=256
)
generated_ids

In [ ]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
output_ids

In [ ]:
tokenizer.decode(output_ids, skip_special_tokens=True)

In [ ]:
# Get the logits for the last token (next word prediction)
next_token_logits = outputs.logits[0, -1, :]

# Apply softmax to convert logits to probabilities
probabilities = torch.softmax(next_token_logits, dim=-1)

# Get the top 10 token IDs with highest probabilities
top_k = 10
top_token_ids = torch.topk(probabilities, top_k).indices

# Decode and display the top 10 predicted tokens
print(f"Top {top_k} predicted tokens:\n")
for i, token_id in enumerate(top_token_ids, 1):
    token = tokenizer.decode([token_id])
    prob = probabilities[token_id].item()
    logit_value = next_token_logits[token_id].item()
    print(f"{i}. '{token}' - Probability: {prob:.4f} ({prob*100:.2f}%), Logit: {logit_value:.2f}")

print(f"\nSum of all probabilities: {probabilities.sum().item():.6f}")